Silver Layer- Data Transformation
Clean and Transform Parcel Raw data into Structured Table

In [0]:
%python
table_exists = spark.catalog.tableExists("parcels_project.silver.property_cleaned")

if table_exists:
    result = spark.sql("""
        SELECT MAX(ingest_timestamp) AS last_timestamp
        FROM parcels_project.silver.property_cleaned
    """).collect()
    
    last_watermark = result[0]["last_timestamp"]
else:
    last_watermark = None

print(f"Last watermark: {last_watermark}")

if last_watermark is not None:
    df_bronze = spark.read.table("parcels_project.bronze.property_raw") \
        .where(f"ingest_timestamp > TIMESTAMP('{last_watermark}')")
else:

    df_bronze = spark.read.table("parcels_project.bronze.property_raw")

print(f"Rows to process this run: {df_bronze.count()}")
display(df_bronze.limit(5))

Last watermark: 2026-09-03 21:27:42.666874
Rows to process this run: 0


OBJECTID,PIN_NUM,CALC_AREA,REID,MAP_NAME,OWNER,ADDR1,ADDR2,ADDR3,DEED_BOOK,DEED_PAGE,DEED_DATE,DEED_ACRES,BLDG_VAL,LAND_VAL,TOTAL_VALUE_ASSD,BILLCLASS,BILLING_CLASS_DECODE,PROPDESC,HEATEDAREA,STNAME,STYPE,STPRE,STSUF,STNUM,STMISC,SITE_ADDRESS,FULL_STREET_NAME,CITY,CITY_DECODE,PLANNING_JURISDICTION,TOWNSHIP,TOWNSHIP_DECODE,FIREDIST,YEAR_BUILT,TOTSALPRICE,SALE_DATE,TYPE_AND_USE,TYPE_USE_DECODE,DESIGNSTYL,DESIGN_STYLE_DECODE,UNITS,LAND_CLASS,LAND_CLASS_DECODE,EXEMPTDESC,EXEMPTSTAT,OWNERSHIP,ACTIVITY,FUNCTION,STRUCTURE,SITE,TOTSTRUCTS,TOTUNITS,OLD_PARCEL_NUMBER,ZIPNUM,PARCEL_PK,LAND_CODE,SHAPEAREA,SHAPELEN,ingest_timestamp,source_file


In [0]:
%python
df_silver = df_bronze
df_silver.createOrReplaceTempView("silver_raw")

In [0]:
SELECT
    COUNT(*) AS row_count
FROM silver_raw;

row_count
0


In [0]:
SELECT COUNT(*) AS row_count, COUNT(DISTINCT PARCEL_PK) AS distinct_pk_count
FROM parcels_project.bronze.property_raw;

row_count,distinct_pk_count
437531,437411


In [0]:
SELECT PARCEL_PK, COUNT(*) AS cnt
FROM parcels_project.bronze.property_raw
GROUP BY PARCEL_PK
HAVING COUNT(*) > 1
ORDER BY cnt DESC;

PARCEL_PK,cnt
null,120


In [0]:
CREATE OR REPLACE TEMP VIEW step1_deduped AS
SELECT * EXCEPT(rn)
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY PARCEL_PK
      ORDER BY ingest_timestamp DESC, OBJECTID DESC
    ) AS rn
  FROM parcels_project.bronze.property_raw
)
WHERE rn = 1;

In [0]:
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT PARCEL_PK) AS distinct_pk
FROM  step1_deduped;

total_rows,distinct_pk
437412,437411


-- ADDR3 dropped per business requirement (not used in analysis)
  -- STPRE dropped per business requirement (not used in analysis)
  -- STSUF dropped per business requirement (not used in analysis)
 -- OLD_PARCEL_NUMBER: keep only complete format ###-#####-####, everything else (bare '--' or partial '542--') becomes NULL

In [0]:
CREATE OR REPLACE TEMP VIEW step2_trimmed AS
SELECT
  OBJECTID,
  PIN_NUM,
  CALC_AREA,
  REID,
  MAP_NAME,
  CASE WHEN TRIM(OWNER) = '' OR LOWER(TRIM(OWNER)) = 'null' THEN NULL ELSE TRIM(OWNER) END AS OWNER,
  CASE WHEN TRIM(ADDR1) = '' OR LOWER(TRIM(ADDR1)) = 'null' THEN NULL ELSE TRIM(ADDR1) END AS ADDR1,
  CASE WHEN TRIM(ADDR2) = '' OR LOWER(TRIM(ADDR2)) = 'null' THEN NULL ELSE TRIM(ADDR2) END AS ADDR2,

  DEED_BOOK,
  DEED_PAGE,
  DEED_DATE,
  DEED_ACRES,
  BLDG_VAL,
  LAND_VAL,
  TOTAL_VALUE_ASSD,
  BILLCLASS,
  CASE WHEN TRIM(BILLING_CLASS_DECODE) = '' OR LOWER(TRIM(BILLING_CLASS_DECODE)) = 'null' THEN NULL ELSE TRIM(BILLING_CLASS_DECODE) END AS BILLING_CLASS_DECODE,
  CASE WHEN TRIM(PROPDESC) = '' OR LOWER(TRIM(PROPDESC)) = 'null' THEN NULL ELSE TRIM(PROPDESC) END AS PROPDESC,
  HEATEDAREA,
  CASE WHEN TRIM(STNAME) = '' OR LOWER(TRIM(STNAME)) = 'null' THEN NULL ELSE TRIM(STNAME) END AS STNAME,
  CASE WHEN TRIM(STYPE) = '' OR LOWER(TRIM(STYPE)) = 'null' THEN NULL ELSE TRIM(STYPE) END AS STYPE,

  STNUM,
  CASE WHEN TRIM(STMISC) = '' OR LOWER(TRIM(STMISC)) = 'null' THEN NULL ELSE TRIM(STMISC) END AS STMISC,
  CASE WHEN TRIM(SITE_ADDRESS) = '' OR LOWER(TRIM(SITE_ADDRESS)) = 'null' THEN NULL ELSE TRIM(SITE_ADDRESS) END AS SITE_ADDRESS,
  CASE WHEN TRIM(FULL_STREET_NAME) = '' OR LOWER(TRIM(FULL_STREET_NAME)) = 'null' THEN NULL ELSE TRIM(FULL_STREET_NAME) END AS FULL_STREET_NAME,
  UPPER(CASE WHEN TRIM(CITY) = '' OR LOWER(TRIM(CITY)) = 'null' THEN NULL ELSE TRIM(CITY) END) AS CITY,
  CASE WHEN TRIM(CITY_DECODE) = '' OR LOWER(TRIM(CITY_DECODE)) = 'null' THEN NULL ELSE TRIM(CITY_DECODE) END AS CITY_DECODE,
  CASE WHEN TRIM(PLANNING_JURISDICTION) = '' OR LOWER(TRIM(PLANNING_JURISDICTION)) = 'null' THEN NULL ELSE TRIM(PLANNING_JURISDICTION) END AS PLANNING_JURISDICTION,
  TOWNSHIP,
  CASE WHEN TRIM(TOWNSHIP_DECODE) = '' OR LOWER(TRIM(TOWNSHIP_DECODE)) = 'null' THEN NULL ELSE TRIM(TOWNSHIP_DECODE) END AS TOWNSHIP_DECODE,
  CASE WHEN TRIM(FIREDIST) = '' OR LOWER(TRIM(FIREDIST)) = 'null' THEN NULL ELSE TRIM(FIREDIST) END AS FIREDIST,
  YEAR_BUILT,
  TOTSALPRICE,
  SALE_DATE,
  TYPE_AND_USE,
  CASE WHEN TRIM(TYPE_USE_DECODE) = '' OR LOWER(TRIM(TYPE_USE_DECODE)) = 'null' THEN NULL ELSE TRIM(TYPE_USE_DECODE) END AS TYPE_USE_DECODE,
  DESIGNSTYL,
  CASE WHEN TRIM(DESIGN_STYLE_DECODE) = '' OR LOWER(TRIM(DESIGN_STYLE_DECODE)) = 'null' THEN NULL ELSE TRIM(DESIGN_STYLE_DECODE) END AS DESIGN_STYLE_DECODE,
  UNITS,
  LAND_CLASS,
  CASE WHEN TRIM(LAND_CLASS_DECODE) = '' OR LOWER(TRIM(LAND_CLASS_DECODE)) = 'null' THEN NULL ELSE TRIM(LAND_CLASS_DECODE) END AS LAND_CLASS_DECODE,
  CASE WHEN TRIM(EXEMPTDESC) = '' OR LOWER(TRIM(EXEMPTDESC)) = 'null' THEN NULL ELSE TRIM(EXEMPTDESC) END AS EXEMPTDESC,
  CASE WHEN TRIM(EXEMPTSTAT) = '' OR LOWER(TRIM(EXEMPTSTAT)) = 'null' THEN NULL ELSE TRIM(EXEMPTSTAT) END AS EXEMPTSTAT,
  CASE WHEN TRIM(OWNERSHIP) = '' OR LOWER(TRIM(OWNERSHIP)) = 'null' THEN NULL ELSE TRIM(OWNERSHIP) END AS OWNERSHIP,
  CASE WHEN TRIM(ACTIVITY) = '' OR LOWER(TRIM(ACTIVITY)) = 'null' THEN NULL ELSE TRIM(ACTIVITY) END AS ACTIVITY,
  CASE WHEN TRIM(FUNCTION) = '' OR LOWER(TRIM(FUNCTION)) = 'null' THEN NULL ELSE TRIM(FUNCTION) END AS FUNCTION,
  CASE WHEN TRIM(STRUCTURE) = '' OR LOWER(TRIM(STRUCTURE)) = 'null' THEN NULL ELSE TRIM(STRUCTURE) END AS STRUCTURE,
  CASE WHEN TRIM(SITE) = '' OR LOWER(TRIM(SITE)) = 'null' THEN NULL ELSE TRIM(SITE) END AS SITE,
  TOTSTRUCTS,
  TOTUNITS,

  CASE
    WHEN TRIM(CAST(OLD_PARCEL_NUMBER AS STRING)) RLIKE '^[0-9]{3}-[0-9]{5}-[0-9]{4}$'
    THEN TRIM(CAST(OLD_PARCEL_NUMBER AS STRING))
    ELSE NULL
  END AS OLD_PARCEL_NUMBER,
  ZIPNUM,
  PARCEL_PK,
  CASE WHEN TRIM(LAND_CODE) = '' OR LOWER(TRIM(LAND_CODE)) = 'null' THEN NULL ELSE TRIM(LAND_CODE) END AS LAND_CODE,
  SHAPEAREA,
  SHAPELEN,
  ingest_timestamp,
  source_file
FROM step1_deduped;

In [0]:
SELECT
  COUNT(*) AS total_rows,
  COUNT(OLD_PARCEL_NUMBER) AS valid_old_parcel_numbers,
  SUM(CASE WHEN OLD_PARCEL_NUMBER IS NULL THEN 1 ELSE 0 END) AS nulled_out
FROM step2_trimmed;

total_rows,valid_old_parcel_numbers,nulled_out
437412,192024,245388


In [0]:
CREATE OR REPLACE TEMP VIEW silver_step3 AS

SELECT *
FROM step2_trimmed
WHERE PARCEL_PK IS NOT NULL
  AND TRIM(CAST(PARCEL_PK AS STRING)) <> '';

In [0]:
SELECT COUNT(*) AS total_rows, COUNT(PARCEL_PK) AS non_null_pk,
    COUNT(DISTINCT PARCEL_PK) AS distinct_pk
    from silver_step3

total_rows,non_null_pk,distinct_pk
437411,437411,437411


In [0]:
%python
df_silver_step3 = spark.table("silver_step3")
display(df_silver_step3.limit(5))

OBJECTID,PIN_NUM,CALC_AREA,REID,MAP_NAME,OWNER,ADDR1,ADDR2,DEED_BOOK,DEED_PAGE,DEED_DATE,DEED_ACRES,BLDG_VAL,LAND_VAL,TOTAL_VALUE_ASSD,BILLCLASS,BILLING_CLASS_DECODE,PROPDESC,HEATEDAREA,STNAME,STYPE,STNUM,STMISC,SITE_ADDRESS,FULL_STREET_NAME,CITY,CITY_DECODE,PLANNING_JURISDICTION,TOWNSHIP,TOWNSHIP_DECODE,FIREDIST,YEAR_BUILT,TOTSALPRICE,SALE_DATE,TYPE_AND_USE,TYPE_USE_DECODE,DESIGNSTYL,DESIGN_STYLE_DECODE,UNITS,LAND_CLASS,LAND_CLASS_DECODE,EXEMPTDESC,EXEMPTSTAT,OWNERSHIP,ACTIVITY,FUNCTION,STRUCTURE,SITE,TOTSTRUCTS,TOTUNITS,OLD_PARCEL_NUMBER,ZIPNUM,PARCEL_PK,LAND_CODE,SHAPEAREA,SHAPELEN,ingest_timestamp,source_file
65404,1726721881,0.32122617,113479,1726 15,"LEE, SANG WOO LEE, ANN CHOI",7832 HARDWICK DR,RALEIGH NC 27615-5413,3349,503,1984/09/12 00:00:00+00,0.32,296037,165000,461037,2,Individual,LO4 REV BLKA HOPE VALL PH1 BM1984 -01030,2546,LAVISTA,CT,4605,null,4605 LAVISTA CT,LAVISTA CT,RAL,RALEIGH,RA,13,Neuse,null,1984,109500,1984/09/12 00:00:00+00,2,TWOFAM,DUP,Duplex,2,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1121,6100,1,2,410-00000-0536,27616,3,R,13992.61209,517.6610918,2026-09-03T21:25:51.832Z,Property.csv
194672,795753828,0.35595309,90033,0795 11,"OLIVE, AARON M OLIVE, LINDSEY B",721 BLENHEIM DR,RALEIGH NC 27612-4902,19930,1051,2025/06/12 00:00:00+00,0.36,322074,630000,952074,2,Individual,LO45 BLENHEIM PLACE SEIII,2681,BLENHEIM,DR,721,null,721 BLENHEIM DR,BLENHEIM DR,RAL,RALEIGH,RA,1,Raleigh,null,1974,960000,2025/06/12 00:00:00+00,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1110,6100,1,1,434-00000-0243,27612,4,R,15505.31639,524.7189471,2026-09-03T21:25:51.832Z,Property.csv
155215,658283916,0.24850344,273966,0658 05,"JOHNSON, DERON JOHNSON, SHELLI",100 PERSIMMON BOTTOM CT,HOLLY SPRINGS NC 27540-7018,16925,2569,2017/09/29 00:00:00+00,0.25,268351,120000,388351,2,Individual,LO137 VALLEYFIELD SUB PH7 BM2000 -782,2038,PERSIMMON BOTTOM,CT,100,null,100 PERSIMMON BOTTOM CT,PERSIMMON BOTTOM CT,HOL,HOLLY SPRINGS,HS,6,Holly Springs,null,2001,253000,2017/09/29 00:00:00+00,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1110,6100,1,1,138-00000-0000,27540,18,R,10824.80992,433.1329581,2026-09-03T21:25:51.832Z,Property.csv
365143,657372315,7.66976165,67944,0657 01,CORPORATION FOR EDUCATIONAL PARTNERSHIPS /TRDG SOUTHERN WAKE ACADEMY,5108 OLD POWELL RD,HOLLY SPRINGS NC 27540-9200,19172,21,2022/10/13 00:00:00+00,7.7,0,189475,189475,1,Business,LO1 NEAL RICHARDSON HEIRS BM1949 -00072,null,SPENCE FARM,RD,0,null,0 SPENCE FARM RD,SPENCE FARM RD,null,null,FV,6,Holly Springs,23,null,3000,1968/01/01 00:00:00+00,null,null,null,null,0,VAC,Vacant,null,null,1020,9000,9900,9000,1000,1,0,null,27540,23,V,334094.8175,2398.112593,2026-09-03T21:25:51.832Z,Property.csv
63344,697144636,0.5455058,247402,0697 03,"O'DEKIRK, JAMES M O'DEKIRK, KATHRYN R",5909 QUAIL MARSH DR,WILLOW SPRING NC 27592-7878,8492,1897,1999/12/29 00:00:00+00,0.55,288081,90000,378081,2,Individual,LO61 BL99 BUWILLOW BM1998 -1311,1662,QUAIL MARSH,DR,5909,null,5909 QUAIL MARSH DR,QUAIL MARSH DR,null,null,WC,12,Middle Creek,23,1999,145000,1999/12/29 00:00:00+00,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1110,6100,1,1,824-00000-0000,27592,35,R,23762.23271,605.5874411,2026-09-03T21:25:51.832Z,Property.csv


In [0]:
CREATE OR REPLACE TEMP VIEW silver_step4 AS
SELECT
  OBJECTID,
  PIN_NUM,
  TRY_CAST(CALC_AREA AS DOUBLE) AS CALC_AREA,
  REID,
  MAP_NAME,
  OWNER, ADDR1, ADDR2,
  DEED_BOOK, DEED_PAGE,
  CAST(TRY_TO_TIMESTAMP(DEED_DATE, 'yyyy/MM/dd HH:mm:ssX') AS DATE) AS DEED_DATE,
  TRY_CAST(DEED_ACRES AS DOUBLE) AS DEED_ACRES,
  TRY_CAST(BLDG_VAL AS DECIMAL(14,2)) AS BLDG_VAL,
  TRY_CAST(LAND_VAL AS DECIMAL(14,2)) AS LAND_VAL,
  TRY_CAST(TOTAL_VALUE_ASSD AS DECIMAL(14,2)) AS TOTAL_VALUE_ASSD,
  BILLCLASS, BILLING_CLASS_DECODE, PROPDESC,
  TRY_CAST(HEATEDAREA AS DOUBLE) AS HEATEDAREA,
  STNAME, STYPE,
  TRY_CAST(STNUM AS INT) AS STNUM,
  STMISC, SITE_ADDRESS, FULL_STREET_NAME,
  CITY, CITY_DECODE, PLANNING_JURISDICTION,
  TOWNSHIP, TOWNSHIP_DECODE, FIREDIST,
  TRY_CAST(YEAR_BUILT AS INT) AS YEAR_BUILT,
  TRY_CAST(TOTSALPRICE AS DECIMAL(14,2)) AS TOTSALPRICE,
  CAST(TRY_TO_TIMESTAMP(SALE_DATE, 'yyyy/MM/dd HH:mm:ssX') AS DATE) AS SALE_DATE,
  TYPE_AND_USE, TYPE_USE_DECODE,
  DESIGNSTYL, DESIGN_STYLE_DECODE,
  TRY_CAST(UNITS AS INT) AS UNITS,
  LAND_CLASS, LAND_CLASS_DECODE,
  EXEMPTDESC, EXEMPTSTAT, OWNERSHIP, ACTIVITY, FUNCTION, STRUCTURE, SITE,
  TRY_CAST(TOTSTRUCTS AS INT) AS TOTSTRUCTS,
  TRY_CAST(TOTUNITS AS INT) AS TOTUNITS,
  OLD_PARCEL_NUMBER,
  LPAD(TRIM(CAST(ZIPNUM AS STRING)), 5, '0') AS ZIPNUM,
  PARCEL_PK,
  LAND_CODE,
  TRY_CAST(SHAPEAREA AS DOUBLE) AS SHAPEAREA,
  TRY_CAST(SHAPELEN AS DOUBLE) AS SHAPELEN,
  ingest_timestamp,
  source_file
FROM silver_step3;

In [0]:
%python
df_silver_step4= spark.table("silver_step4")
display(df_silver_step4.limit(5))

OBJECTID,PIN_NUM,CALC_AREA,REID,MAP_NAME,OWNER,ADDR1,ADDR2,DEED_BOOK,DEED_PAGE,DEED_DATE,DEED_ACRES,BLDG_VAL,LAND_VAL,TOTAL_VALUE_ASSD,BILLCLASS,BILLING_CLASS_DECODE,PROPDESC,HEATEDAREA,STNAME,STYPE,STNUM,STMISC,SITE_ADDRESS,FULL_STREET_NAME,CITY,CITY_DECODE,PLANNING_JURISDICTION,TOWNSHIP,TOWNSHIP_DECODE,FIREDIST,YEAR_BUILT,TOTSALPRICE,SALE_DATE,TYPE_AND_USE,TYPE_USE_DECODE,DESIGNSTYL,DESIGN_STYLE_DECODE,UNITS,LAND_CLASS,LAND_CLASS_DECODE,EXEMPTDESC,EXEMPTSTAT,OWNERSHIP,ACTIVITY,FUNCTION,STRUCTURE,SITE,TOTSTRUCTS,TOTUNITS,OLD_PARCEL_NUMBER,ZIPNUM,PARCEL_PK,LAND_CODE,SHAPEAREA,SHAPELEN,ingest_timestamp,source_file
65404,1726721881,0.32122617,113479,1726 15,"LEE, SANG WOO LEE, ANN CHOI",7832 HARDWICK DR,RALEIGH NC 27615-5413,3349,503,1984-09-12,0.32,296037.00,165000.00,461037.00,2,Individual,LO4 REV BLKA HOPE VALL PH1 BM1984 -01030,2546.0,LAVISTA,CT,4605,null,4605 LAVISTA CT,LAVISTA CT,RAL,RALEIGH,RA,13,Neuse,null,1984,109500.00,1984-09-12,2,TWOFAM,DUP,Duplex,2,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1121,6100,1,2,410-00000-0536,27616,3,R,13992.61209,517.6610918,2026-09-03T21:25:51.832Z,Property.csv
194672,795753828,0.35595309,90033,0795 11,"OLIVE, AARON M OLIVE, LINDSEY B",721 BLENHEIM DR,RALEIGH NC 27612-4902,19930,1051,2025-06-12,0.36,322074.00,630000.00,952074.00,2,Individual,LO45 BLENHEIM PLACE SEIII,2681.0,BLENHEIM,DR,721,null,721 BLENHEIM DR,BLENHEIM DR,RAL,RALEIGH,RA,1,Raleigh,null,1974,960000.00,2025-06-12,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1110,6100,1,1,434-00000-0243,27612,4,R,15505.31639,524.7189471,2026-09-03T21:25:51.832Z,Property.csv
155215,658283916,0.24850344,273966,0658 05,"JOHNSON, DERON JOHNSON, SHELLI",100 PERSIMMON BOTTOM CT,HOLLY SPRINGS NC 27540-7018,16925,2569,2017-09-29,0.25,268351.00,120000.00,388351.00,2,Individual,LO137 VALLEYFIELD SUB PH7 BM2000 -782,2038.0,PERSIMMON BOTTOM,CT,100,null,100 PERSIMMON BOTTOM CT,PERSIMMON BOTTOM CT,HOL,HOLLY SPRINGS,HS,6,Holly Springs,null,2001,253000.00,2017-09-29,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1110,6100,1,1,138-00000-0000,27540,18,R,10824.80992,433.1329581,2026-09-03T21:25:51.832Z,Property.csv
365143,657372315,7.66976165,67944,0657 01,CORPORATION FOR EDUCATIONAL PARTNERSHIPS /TRDG SOUTHERN WAKE ACADEMY,5108 OLD POWELL RD,HOLLY SPRINGS NC 27540-9200,19172,21,2022-10-13,7.7,0.00,189475.00,189475.00,1,Business,LO1 NEAL RICHARDSON HEIRS BM1949 -00072,null,SPENCE FARM,RD,0,null,0 SPENCE FARM RD,SPENCE FARM RD,null,null,FV,6,Holly Springs,23,null,3000.00,1968-01-01,null,null,null,null,0,VAC,Vacant,null,null,1020,9000,9900,9000,1000,1,0,null,27540,23,V,334094.8175,2398.112593,2026-09-03T21:25:51.832Z,Property.csv
63344,697144636,0.5455058,247402,0697 03,"O'DEKIRK, JAMES M O'DEKIRK, KATHRYN R",5909 QUAIL MARSH DR,WILLOW SPRING NC 27592-7878,8492,1897,1999-12-29,0.55,288081.00,90000.00,378081.00,2,Individual,LO61 BL99 BUWILLOW BM1998 -1311,1662.0,QUAIL MARSH,DR,5909,null,5909 QUAIL MARSH DR,QUAIL MARSH DR,null,null,WC,12,Middle Creek,23,1999,145000.00,1999-12-29,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,1010,1100,1100,1110,6100,1,1,824-00000-0000,27592,35,R,23762.23271,605.5874411,2026-09-03T21:25:51.832Z,Property.csv


In [0]:
SELECT
  SUM(CASE WHEN DEED_DATE IS NULL THEN 1 ELSE 0 END) AS deed_date_nulls,
  SUM(CASE WHEN SALE_DATE IS NULL THEN 1 ELSE 0 END) AS sale_date_nulls,
  COUNT(*) AS total_rows
FROM silver_step4;

deed_date_nulls,sale_date_nulls,total_rows
135,80279,437411


In [0]:
CREATE OR REPLACE TEMP VIEW silver_step5 AS
SELECT
  * EXCEPT (CITY_DECODE, TOWNSHIP_DECODE, TYPE_USE_DECODE, LAND_CLASS_DECODE),
  COALESCE(CITY_DECODE, CITY, 'UNKNOWN')  AS CITY_DECODE,
  COALESCE(TOWNSHIP_DECODE, 'UNKNOWN')    AS TOWNSHIP_DECODE,
  COALESCE(TYPE_USE_DECODE, 'UNKNOWN')    AS TYPE_USE_DECODE,
  COALESCE(LAND_CLASS_DECODE, 'UNKNOWN')  AS LAND_CLASS_DECODE,
  CASE WHEN EXEMPTSTAT IS NOT NULL THEN TRUE ELSE FALSE END AS IS_TAX_EXEMPT,
  TRIM(CONCAT_WS(' ', STNUM, STNAME, STYPE, STMISC)) AS STREET_ADDRESS_FULL
FROM silver_step4;

In [0]:
%python
df_silver_step5= spark.table("silver_step5")
display(df_silver_step5.limit(5))

OBJECTID,PIN_NUM,CALC_AREA,REID,MAP_NAME,OWNER,ADDR1,ADDR2,DEED_BOOK,DEED_PAGE,DEED_DATE,DEED_ACRES,BLDG_VAL,LAND_VAL,TOTAL_VALUE_ASSD,BILLCLASS,BILLING_CLASS_DECODE,PROPDESC,HEATEDAREA,STNAME,STYPE,STNUM,STMISC,SITE_ADDRESS,FULL_STREET_NAME,CITY,PLANNING_JURISDICTION,TOWNSHIP,FIREDIST,YEAR_BUILT,TOTSALPRICE,SALE_DATE,TYPE_AND_USE,DESIGNSTYL,DESIGN_STYLE_DECODE,UNITS,LAND_CLASS,EXEMPTDESC,EXEMPTSTAT,OWNERSHIP,ACTIVITY,FUNCTION,STRUCTURE,SITE,TOTSTRUCTS,TOTUNITS,OLD_PARCEL_NUMBER,ZIPNUM,PARCEL_PK,LAND_CODE,SHAPEAREA,SHAPELEN,ingest_timestamp,source_file,CITY_DECODE,TOWNSHIP_DECODE,TYPE_USE_DECODE,LAND_CLASS_DECODE,IS_TAX_EXEMPT,STREET_ADDRESS_FULL
65404,1726721881,0.32122617,113479,1726 15,"LEE, SANG WOO LEE, ANN CHOI",7832 HARDWICK DR,RALEIGH NC 27615-5413,3349,503,1984-09-12,0.32,296037.00,165000.00,461037.00,2,Individual,LO4 REV BLKA HOPE VALL PH1 BM1984 -01030,2546.0,LAVISTA,CT,4605,null,4605 LAVISTA CT,LAVISTA CT,RAL,RA,13,null,1984,109500.00,1984-09-12,2,DUP,Duplex,2,RHS,null,null,1010,1100,1100,1121,6100,1,2,410-00000-0536,27616,3,R,13992.61209,517.6610918,2026-09-03T21:25:51.832Z,Property.csv,RALEIGH,Neuse,TWOFAM,Residential Less Than 10 Acres,false,4605 LAVISTA CT
194672,795753828,0.35595309,90033,0795 11,"OLIVE, AARON M OLIVE, LINDSEY B",721 BLENHEIM DR,RALEIGH NC 27612-4902,19930,1051,2025-06-12,0.36,322074.00,630000.00,952074.00,2,Individual,LO45 BLENHEIM PLACE SEIII,2681.0,BLENHEIM,DR,721,null,721 BLENHEIM DR,BLENHEIM DR,RAL,RA,1,null,1974,960000.00,2025-06-12,1,CVL,Conventional,1,RHS,null,null,1010,1100,1100,1110,6100,1,1,434-00000-0243,27612,4,R,15505.31639,524.7189471,2026-09-03T21:25:51.832Z,Property.csv,RALEIGH,Raleigh,SINGLFAM,Residential Less Than 10 Acres,false,721 BLENHEIM DR
155215,658283916,0.24850344,273966,0658 05,"JOHNSON, DERON JOHNSON, SHELLI",100 PERSIMMON BOTTOM CT,HOLLY SPRINGS NC 27540-7018,16925,2569,2017-09-29,0.25,268351.00,120000.00,388351.00,2,Individual,LO137 VALLEYFIELD SUB PH7 BM2000 -782,2038.0,PERSIMMON BOTTOM,CT,100,null,100 PERSIMMON BOTTOM CT,PERSIMMON BOTTOM CT,HOL,HS,6,null,2001,253000.00,2017-09-29,1,CVL,Conventional,1,RHS,null,null,1010,1100,1100,1110,6100,1,1,138-00000-0000,27540,18,R,10824.80992,433.1329581,2026-09-03T21:25:51.832Z,Property.csv,HOLLY SPRINGS,Holly Springs,SINGLFAM,Residential Less Than 10 Acres,false,100 PERSIMMON BOTTOM CT
365143,657372315,7.66976165,67944,0657 01,CORPORATION FOR EDUCATIONAL PARTNERSHIPS /TRDG SOUTHERN WAKE ACADEMY,5108 OLD POWELL RD,HOLLY SPRINGS NC 27540-9200,19172,21,2022-10-13,7.7,0.00,189475.00,189475.00,1,Business,LO1 NEAL RICHARDSON HEIRS BM1949 -00072,null,SPENCE FARM,RD,0,null,0 SPENCE FARM RD,SPENCE FARM RD,null,FV,6,23,null,3000.00,1968-01-01,null,null,null,0,VAC,null,null,1020,9000,9900,9000,1000,1,0,null,27540,23,V,334094.8175,2398.112593,2026-09-03T21:25:51.832Z,Property.csv,UNKNOWN,Holly Springs,UNKNOWN,Vacant,false,0 SPENCE FARM RD
63344,697144636,0.5455058,247402,0697 03,"O'DEKIRK, JAMES M O'DEKIRK, KATHRYN R",5909 QUAIL MARSH DR,WILLOW SPRING NC 27592-7878,8492,1897,1999-12-29,0.55,288081.00,90000.00,378081.00,2,Individual,LO61 BL99 BUWILLOW BM1998 -1311,1662.0,QUAIL MARSH,DR,5909,null,5909 QUAIL MARSH DR,QUAIL MARSH DR,null,WC,12,23,1999,145000.00,1999-12-29,1,CVL,Conventional,1,RHS,null,null,1010,1100,1100,1110,6100,1,1,824-00000-0000,27592,35,R,23762.23271,605.5874411,2026-09-03T21:25:51.832Z,Property.csv,UNKNOWN,Middle Creek,SINGLFAM,Residential Less Than 10 Acres,false,5909 QUAIL MARSH DR


In [0]:
CREATE OR REPLACE TEMP VIEW silver_step6 AS
SELECT
    *,
    YEAR(CURRENT_DATE()) - YEAR_BUILT AS PROPERTY_AGE_YEARS
FROM silver_step5;

In [0]:
%python
df_silver_step6 = spark.table("silver_step6")
display(df_silver_step6.limit(5))


OBJECTID,PIN_NUM,CALC_AREA,REID,MAP_NAME,OWNER,ADDR1,ADDR2,DEED_BOOK,DEED_PAGE,DEED_DATE,DEED_ACRES,BLDG_VAL,LAND_VAL,TOTAL_VALUE_ASSD,BILLCLASS,BILLING_CLASS_DECODE,PROPDESC,HEATEDAREA,STNAME,STYPE,STNUM,STMISC,SITE_ADDRESS,FULL_STREET_NAME,CITY,PLANNING_JURISDICTION,TOWNSHIP,FIREDIST,YEAR_BUILT,TOTSALPRICE,SALE_DATE,TYPE_AND_USE,DESIGNSTYL,DESIGN_STYLE_DECODE,UNITS,LAND_CLASS,EXEMPTDESC,EXEMPTSTAT,OWNERSHIP,ACTIVITY,FUNCTION,STRUCTURE,SITE,TOTSTRUCTS,TOTUNITS,OLD_PARCEL_NUMBER,ZIPNUM,PARCEL_PK,LAND_CODE,SHAPEAREA,SHAPELEN,ingest_timestamp,source_file,CITY_DECODE,TOWNSHIP_DECODE,TYPE_USE_DECODE,LAND_CLASS_DECODE,IS_TAX_EXEMPT,STREET_ADDRESS_FULL,PROPERTY_AGE_YEARS
65404,1726721881,0.32122617,113479,1726 15,"LEE, SANG WOO LEE, ANN CHOI",7832 HARDWICK DR,RALEIGH NC 27615-5413,3349,503,1984-09-12,0.32,296037.00,165000.00,461037.00,2,Individual,LO4 REV BLKA HOPE VALL PH1 BM1984 -01030,2546.0,LAVISTA,CT,4605,null,4605 LAVISTA CT,LAVISTA CT,RAL,RA,13,null,1984,109500.00,1984-09-12,2,DUP,Duplex,2,RHS,null,null,1010,1100,1100,1121,6100,1,2,410-00000-0536,27616,3,R,13992.61209,517.6610918,2026-09-03T21:25:51.832Z,Property.csv,RALEIGH,Neuse,TWOFAM,Residential Less Than 10 Acres,false,4605 LAVISTA CT,42
194672,795753828,0.35595309,90033,0795 11,"OLIVE, AARON M OLIVE, LINDSEY B",721 BLENHEIM DR,RALEIGH NC 27612-4902,19930,1051,2025-06-12,0.36,322074.00,630000.00,952074.00,2,Individual,LO45 BLENHEIM PLACE SEIII,2681.0,BLENHEIM,DR,721,null,721 BLENHEIM DR,BLENHEIM DR,RAL,RA,1,null,1974,960000.00,2025-06-12,1,CVL,Conventional,1,RHS,null,null,1010,1100,1100,1110,6100,1,1,434-00000-0243,27612,4,R,15505.31639,524.7189471,2026-09-03T21:25:51.832Z,Property.csv,RALEIGH,Raleigh,SINGLFAM,Residential Less Than 10 Acres,false,721 BLENHEIM DR,52
155215,658283916,0.24850344,273966,0658 05,"JOHNSON, DERON JOHNSON, SHELLI",100 PERSIMMON BOTTOM CT,HOLLY SPRINGS NC 27540-7018,16925,2569,2017-09-29,0.25,268351.00,120000.00,388351.00,2,Individual,LO137 VALLEYFIELD SUB PH7 BM2000 -782,2038.0,PERSIMMON BOTTOM,CT,100,null,100 PERSIMMON BOTTOM CT,PERSIMMON BOTTOM CT,HOL,HS,6,null,2001,253000.00,2017-09-29,1,CVL,Conventional,1,RHS,null,null,1010,1100,1100,1110,6100,1,1,138-00000-0000,27540,18,R,10824.80992,433.1329581,2026-09-03T21:25:51.832Z,Property.csv,HOLLY SPRINGS,Holly Springs,SINGLFAM,Residential Less Than 10 Acres,false,100 PERSIMMON BOTTOM CT,25
365143,657372315,7.66976165,67944,0657 01,CORPORATION FOR EDUCATIONAL PARTNERSHIPS /TRDG SOUTHERN WAKE ACADEMY,5108 OLD POWELL RD,HOLLY SPRINGS NC 27540-9200,19172,21,2022-10-13,7.7,0.00,189475.00,189475.00,1,Business,LO1 NEAL RICHARDSON HEIRS BM1949 -00072,null,SPENCE FARM,RD,0,null,0 SPENCE FARM RD,SPENCE FARM RD,null,FV,6,23,null,3000.00,1968-01-01,null,null,null,0,VAC,null,null,1020,9000,9900,9000,1000,1,0,null,27540,23,V,334094.8175,2398.112593,2026-09-03T21:25:51.832Z,Property.csv,UNKNOWN,Holly Springs,UNKNOWN,Vacant,false,0 SPENCE FARM RD,null
63344,697144636,0.5455058,247402,0697 03,"O'DEKIRK, JAMES M O'DEKIRK, KATHRYN R",5909 QUAIL MARSH DR,WILLOW SPRING NC 27592-7878,8492,1897,1999-12-29,0.55,288081.00,90000.00,378081.00,2,Individual,LO61 BL99 BUWILLOW BM1998 -1311,1662.0,QUAIL MARSH,DR,5909,null,5909 QUAIL MARSH DR,QUAIL MARSH DR,null,WC,12,23,1999,145000.00,1999-12-29,1,CVL,Conventional,1,RHS,null,null,1010,1100,1100,1110,6100,1,1,824-00000-0000,27592,35,R,23762.23271,605.5874411,2026-09-03T21:25:51.832Z,Property.csv,UNKNOWN,Middle Creek,SINGLFAM,Residential Less Than 10 Acres,false,5909 QUAIL MARSH DR,27


In [0]:
SELECT
    OBJECTID,
    YEAR_BUILT,
    PROPERTY_AGE_YEARS
FROM silver_step6
LIMIT 2;

OBJECTID,YEAR_BUILT,PROPERTY_AGE_YEARS
65404,1984,42
194672,1974,52


In [0]:
CREATE OR REPLACE TEMP VIEW silver_step7 AS
SELECT
    *,
    CASE
        WHEN YEAR_BUILT IS NULL THEN 'Unknown'
        WHEN YEAR_BUILT < 1950 THEN 'Pre-1950'
        WHEN YEAR_BUILT < 1980 THEN '1950-1979'
        WHEN YEAR_BUILT < 2000 THEN '1980-1999'
        WHEN YEAR_BUILT < 2010 THEN '2000-2009'
        ELSE '2010+'
    END AS BUILD_ERA
FROM silver_step6;

In [0]:
SELECT
    YEAR_BUILT,
    BUILD_ERA
FROM silver_step7
LIMIT 5;

YEAR_BUILT,BUILD_ERA
1984,1980-1999
1974,1950-1979
2001,2000-2009
null,Unknown
1999,1980-1999


Assessed Value Per Square Foot-Value per Square Foot Calculation.
Assessed value is the dollar amount a local government assigns to a property to calculate property taxes.

In [0]:
CREATE OR REPLACE TEMP VIEW silver_step8 AS
SELECT
    *,
    CASE
        WHEN HEATEDAREA IS NULL OR HEATEDAREA = 0 THEN NULL
        ELSE ROUND(TOTAL_VALUE_ASSD / HEATEDAREA, 2)
    END AS ASSESSED_VALUE_PER_SQFT
FROM silver_step7;

In [0]:
SELECT
    TOTAL_VALUE_ASSD,
    HEATEDAREA,
    ASSESSED_VALUE_PER_SQFT
FROM silver_step8
LIMIT 20;

TOTAL_VALUE_ASSD,HEATEDAREA,ASSESSED_VALUE_PER_SQFT
461037.00,2546.0,181.08
952074.00,2681.0,355.12
388351.00,2038.0,190.55
189475.00,null,null
378081.00,1662.0,227.49
107367.00,864.0,124.27
427450.00,2188.0,195.36
83501.00,856.0,97.55
405670.00,1904.0,213.06
451914.00,2338.0,193.29


In [0]:
CREATE OR REPLACE TEMP VIEW silver_step9 AS
SELECT
    *,
    CASE
        WHEN DEED_ACRES IS NULL OR DEED_ACRES = 0 THEN NULL
        ELSE ROUND(TOTAL_VALUE_ASSD / DEED_ACRES, 2)
    END AS ASSESSED_VALUE_PER_ACRE
FROM silver_step8;

In [0]:
SELECT
    TOTAL_VALUE_ASSD,
    DEED_ACRES,
    ASSESSED_VALUE_PER_ACRE
FROM silver_step9
LIMIT 20;

TOTAL_VALUE_ASSD,DEED_ACRES,ASSESSED_VALUE_PER_ACRE
461037.00,0.32,1440740.63
952074.00,0.36,2644650.0
388351.00,0.25,1553404.0
189475.00,7.7,24607.14
378081.00,0.55,687420.0
107367.00,0.79,135907.59
427450.00,0.28,1526607.14
83501.00,0.79,105697.47
405670.00,0.53,765415.09
451914.00,0.36,1255316.67


In [0]:
CREATE OR REPLACE TEMP VIEW silver_agg_by_city AS
SELECT
    CITY_DECODE,
    COUNT(*) AS PROPERTY_COUNT,
    SUM(TOTAL_VALUE_ASSD) AS TOTAL_ASSESSED_VALUE,
    AVG(TOTAL_VALUE_ASSD) AS AVG_ASSESSED_VALUE,
    SUM(HEATEDAREA) AS TOTAL_HEATED_AREA,
    AVG(ASSESSED_VALUE_PER_SQFT) AS AVG_VALUE_PER_SQFT
FROM silver_step9
GROUP BY CITY_DECODE;

Property Aggregation by City- aggregation transformation combines multiple property rows to produce summary-level information.

In [0]:
SELECT 
CITY_DECODE,
   TOTAL_ASSESSED_VALUE,
   AVG_ASSESSED_VALUE,
    TOTAL_HEATED_AREA,
     AVG_VALUE_PER_SQFT
FROM silver_agg_by_city
LIMIT 5;

CITY_DECODE,TOTAL_ASSESSED_VALUE,AVG_ASSESSED_VALUE,TOTAL_HEATED_AREA,AVG_VALUE_PER_SQFT
RALEIGH,123564083147.00,885091.494255,5.41661391E8,242.3648913000871
CARY,49512814138.00,865865.976566,2.07640789E8,253.04518052120997
WAKE FOREST,11562141152.00,577442.998152,5.8267339E7,200.090449140603
ANGIER,339089419.00,304662.550764,1756633.0,169.23907352941177
DURHAM,258471277.00,694815.260753,1153559.0,202.81749216300943


In [0]:
%sql
SELECT COUNT(*) AS city_count FROM parcels_project.silver.property_agg_by_city;

city_count
16


In [0]:
MERGE INTO parcels_project.silver.property_cleaned AS tgt
USING (
  SELECT
    PARCEL_PK, OBJECTID, PIN_NUM, CALC_AREA, REID, MAP_NAME, OWNER, ADDR1, ADDR2,
    DEED_BOOK, DEED_PAGE, DEED_DATE, DEED_ACRES, BLDG_VAL, LAND_VAL,
    TOTAL_VALUE_ASSD, BILLCLASS, BILLING_CLASS_DECODE, PROPDESC, HEATEDAREA,
    STNAME, STYPE, STNUM, STMISC, SITE_ADDRESS, FULL_STREET_NAME,
    STREET_ADDRESS_FULL, CITY, CITY_DECODE, PLANNING_JURISDICTION, TOWNSHIP,
    TOWNSHIP_DECODE, FIREDIST, YEAR_BUILT, PROPERTY_AGE_YEARS, BUILD_ERA,
    TOTSALPRICE, SALE_DATE, TYPE_AND_USE, TYPE_USE_DECODE, DESIGNSTYL,
    DESIGN_STYLE_DECODE, UNITS, LAND_CLASS, LAND_CLASS_DECODE, EXEMPTDESC,
    EXEMPTSTAT, IS_TAX_EXEMPT, ASSESSED_VALUE_PER_SQFT, ASSESSED_VALUE_PER_ACRE,
    OWNERSHIP, ACTIVITY, FUNCTION, STRUCTURE, SITE, TOTSTRUCTS, TOTUNITS,
    OLD_PARCEL_NUMBER, ZIPNUM, LAND_CODE, SHAPEAREA, SHAPELEN,
    ingest_timestamp, source_file
  FROM silver_step9
) AS src
ON tgt.PARCEL_PK = src.PARCEL_PK
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
437411,437411,0,0


In [0]:
%python
df_property_cleaned = spark.table("parcels_project.silver.property_cleaned")
display(df_property_cleaned.limit(5))

PARCEL_PK,OBJECTID,PIN_NUM,CALC_AREA,REID,MAP_NAME,OWNER,ADDR1,ADDR2,DEED_BOOK,DEED_PAGE,DEED_DATE,DEED_ACRES,BLDG_VAL,LAND_VAL,TOTAL_VALUE_ASSD,BILLCLASS,BILLING_CLASS_DECODE,PROPDESC,HEATEDAREA,STNAME,STYPE,STNUM,STMISC,SITE_ADDRESS,FULL_STREET_NAME,STREET_ADDRESS_FULL,CITY,CITY_DECODE,PLANNING_JURISDICTION,TOWNSHIP,TOWNSHIP_DECODE,FIREDIST,YEAR_BUILT,PROPERTY_AGE_YEARS,BUILD_ERA,TOTSALPRICE,SALE_DATE,TYPE_AND_USE,TYPE_USE_DECODE,DESIGNSTYL,DESIGN_STYLE_DECODE,UNITS,LAND_CLASS,LAND_CLASS_DECODE,EXEMPTDESC,EXEMPTSTAT,IS_TAX_EXEMPT,ASSESSED_VALUE_PER_SQFT,ASSESSED_VALUE_PER_ACRE,OWNERSHIP,ACTIVITY,FUNCTION,STRUCTURE,SITE,TOTSTRUCTS,TOTUNITS,OLD_PARCEL_NUMBER,ZIPNUM,LAND_CODE,SHAPEAREA,SHAPELEN,ingest_timestamp,source_file
1,401626,1706218758,0.35672913,42904,1706 18,"ALLEN, AMANDA F ALLEN, SCOTT J",712 RICHMOND ST,RALEIGH NC 27609-5555,17217,1970,2018-08-17,0.36,346029.00,546250.00,892279.00,2,Individual,LO19 NORTH HILLS SE8-A BL10,2816.0,RICHMOND,ST,712,null,712 RICHMOND ST,RICHMOND ST,712 RICHMOND ST,RAL,RALEIGH,RA,1,Raleigh,null,1961,65,1950-1979,370000.00,2018-08-17,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,false,316.86,2478552.78,1010,1100,1100,1110,6100,1,1,null,27609,R,15539.12098,504.7524686,2026-09-03T21:25:51.832Z,Property.csv
6,418847,1709618708,1.02633194,138813,1709 04,"CUTTER, PATRICK CUTTER, AZURE",10901 ERVIN CT,RALEIGH NC 27614-9540,18058,1060,2020-09-04,1.02,494715.00,200000.00,694715.00,2,Individual,LO52 COBBLE CREEK P 3 BM1984 -01772,3074.0,ERVIN,CT,10901,null,10901 ERVIN CT,ERVIN CT,10901 ERVIN CT,null,UNKNOWN,WC,2,Barton's Creek,23,1986,40,1980-1999,480000.00,2020-09-04,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,false,226.0,681093.14,1010,1100,1100,1110,6100,1,1,260-00000-0404,27614,R,44707.0191,879.2607874,2026-09-03T21:25:51.832Z,Property.csv
37,279707,1739858005,0.70734522,113480,1739 02,"FLOWERS, JOHN CAMERON",1001 LIGHTFOOT CT,WAKE FOREST NC 27587-9342,19387,1753,2023-07-19,0.67,239262.00,85000.00,324262.00,2,Individual,LO7 A COUNTRY PLACE PHI BM1980 -230,1369.0,LIGHTFOOT,CT,1001,null,1001 LIGHTFOOT CT,LIGHTFOOT CT,1001 LIGHTFOOT CT,null,UNKNOWN,WF,19,Wake Forest,23,1983,43,1980-1999,320000.00,2023-07-19,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,false,236.86,483973.13,1010,1100,1100,1110,6100,1,1,265-00000-0038,27587,R,30811.95778,752.5888419,2026-09-03T21:25:51.832Z,Property.csv
66,58519,795757837,0.31915868,90037,0795 12,"WESSEL, CORY M WESSEL, LINDSAY",705 BLENHEIM DR,RALEIGH NC 27612-4902,16511,1402,2016-08-29,0.32,358671.00,630000.00,988671.00,2,Individual,LO49 BLENHEIM PLACE SEIII BM1974 -279,2785.0,BLENHEIM,DR,705,null,705 BLENHEIM DR,BLENHEIM DR,705 BLENHEIM DR,RAL,RALEIGH,RA,1,Raleigh,null,1975,51,1950-1979,620000.00,2016-08-29,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,false,355.0,3089596.88,1010,1100,1100,1110,6100,1,1,434-00000-0247,27612,R,13902.55201,500.0204667,2026-09-03T21:25:51.832Z,Property.csv
82,415709,1739848952,0.69504146,113483,1739 04,"BRODERS, SCOTT T",9505 LIGON MILL RD,WAKE FOREST NC 27587-7717,19869,2510,2025-04-03,0.68,218113.00,85000.00,303113.00,2,Individual,LO8 A COUNTRY PLACE SUB PHI BM1980 -230,1308.0,LIGON MILL,RD,9505,null,9505 LIGON MILL RD,LIGON MILL RD,9505 LIGON MILL RD,null,UNKNOWN,WF,19,Wake Forest,23,1983,43,1980-1999,82000.00,2009-06-22,1,SINGLFAM,CVL,Conventional,1,RHS,Residential Less Than 10 Acres,null,null,false,231.74,445754.41,1010,1100,1100,1110,6100,1,1,265-00000-0039,27587,R,30276.00583,756.7906446,2026-09-03T21:25:51.832Z,Property.csv


In [0]:
%python
print("Total columns:", len(df_property_cleaned.columns))

Total columns: 64


In [0]:
SELECT source_file, COUNT(*) AS row_count
FROM parcels_project.silver.property_cleaned
GROUP BY source_file
ORDER BY row_count DESC;

source_file,row_count
Property.csv,437403
new_property1.csv,5
new_parcel6.csv,3
